# 01 Data Download

Mount Google Drive, install the repo, and persist raw NSE intraday and daily-context data as Parquet. The primary path is the repo's `openchart` NSE charting adapter; uploaded Parquet/CSV data is also supported for real historical intraday training data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

for path in (DATA_DIR / 'raw', DATA_DIR / 'processed', MODEL_DIR, LOG_DIR, REPORT_DIR, ARTIFACT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Copy or clone this repo to {REPO_DIR} before running this notebook')

%cd $REPO_DIR

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
from dataclasses import replace
from pathlib import Path

import pandas as pd

from config import MarketConfig, PathConfig
from data.loader import MarketDataLoader, standardize_ohlcv

SYMBOL = 'RELIANCE'
TICKER = 'RELIANCE.NS'
SERIES = 'EQ'
INTERVAL = '5m'
LOOKBACK_DAYS = 365

# Preferred for real training: place a historical 5-minute Parquet/CSV here.
# Expected columns: datetime/date/index, open, high, low, close, volume.
LOCAL_INTRADAY_FILE = DATA_DIR / 'raw' / 'RELIANCE_NS_5m_source.parquet'

# If openchart returns no data, this notebook will stop here by default.
# Put a real broker/vendor intraday file at LOCAL_INTRADAY_FILE for serious training.
# Keep False for real training. Set True only for a short yfinance smoke test.
ALLOW_YFINANCE_5M_SMOKE = False

paths = PathConfig(
    root=BASE,
    raw_data_dir=DATA_DIR / 'raw',
    processed_data_dir=DATA_DIR / 'processed',
    artifact_dir=ARTIFACT_DIR,
    model_artifact_dir=MODEL_DIR,
    report_dir=REPORT_DIR,
)
market = MarketConfig(
    symbol=SYMBOL,
    ticker=TICKER,
    series=SERIES,
    interval=INTERVAL,
    intraday_source='openchart',
    lookback_days=LOOKBACK_DAYS,
)
loader = MarketDataLoader(paths, market)

if LOCAL_INTRADAY_FILE.exists():
    if LOCAL_INTRADAY_FILE.suffix.lower() == '.csv':
        raw_intraday = pd.read_csv(LOCAL_INTRADAY_FILE)
    else:
        raw_intraday = pd.read_parquet(LOCAL_INTRADAY_FILE)
    intraday = standardize_ohlcv(raw_intraday, intraday=True, timezone=market.timezone)
    intraday.to_parquet(loader.intraday_path())
    print(f'Loaded uploaded intraday data: {loader.intraday_path()} rows={len(intraday)}')
else:
    try:
        result = loader.download_intraday(force_refresh=False, source='openchart')
        print(f'Downloaded intraday data: {result.path} rows={result.rows} refreshed={result.refreshed}')
    except Exception as exc:
        print(f'Primary intraday adapter failed: {exc}')
        if not ALLOW_YFINANCE_5M_SMOKE:
            raise RuntimeError('openchart could not fetch intraday data. Try a shorter LOOKBACK_DAYS, provide historical 5m data at LOCAL_INTRADAY_FILE, or enable ALLOW_YFINANCE_5M_SMOKE for a short smoke test') from exc
        smoke_market = replace(market, intraday_source='yfinance-5m', lookback_days=60)
        loader = MarketDataLoader(paths, smoke_market)
        result = loader.download_intraday(force_refresh=False, source='yfinance-5m')
        print(f'Yfinance smoke intraday data: {result.path} rows={result.rows} refreshed={result.refreshed}')

daily_result = loader.download_daily_context(force_refresh=False)
print(f'Daily context data: {daily_result.path} rows={daily_result.rows} refreshed={daily_result.refreshed}')